In [33]:
%load_ext autoreload
%autoreload 2
# %matplotlib inline

import os
while 'notebooks' in os.getcwd():
    os.chdir("../")

import torch
from torch import nn, einsum

import quantus
import gc
import torch.nn.functional as F
import pandas as pd

from lib.helpers import plot_example_grid
from lib.attributions import GradientAscentDiff, PullbackAscentDiff, DoublePullbackAscentDiff, \
    quantus_pullback_ascent_diff_explain_func, quantus_double_pullback_ascent_diff_explain_func
from lib.setup import setup_notebook
from lib.defaults import get_default_kwargs
from lib.surrogates import LayerNorm2d, PVTAttention, soften_module_inplace_
from lib.evaluator import QuantusEvaluator, default_explainers, default_metrics

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
loaded_df_resnet_gc = QuantusEvaluator.load_results(f"results/quantus_resnet_20_25_gc")
print("num_samples:", len(loaded_df_resnet_gc.iloc[0].iloc[0]))
summary_resnet_gc = QuantusEvaluator.summarize_results(loaded_df_resnet_gc)
summary_resnet_gc

num_samples: 500


,infidelity,faithfulness_correlation,monotonicity_correlation,faithfulness_estimate,max_sensitivity,sparseness,random_logit
SoftPullback,64.948±53.29,0.1±0.104,0.536±0.428,0.542±0.168,0.16±0.049,0.628±0.054,-0.083±0.389
GuidedGradCam,2.63e+05±4.74e+05,0.039±0.106,0.593±0.379,0.173±0.148,0.271±0.103,0.812±0.067,0.415±0.328


In [35]:
loaded_df_vgg_gc = QuantusEvaluator.load_results(f"results/quantus_vgg_20_25_gc")
print("num_samples:", len(loaded_df_vgg_gc.iloc[0].iloc[0]))
summary_vgg_gc = QuantusEvaluator.summarize_results(loaded_df_vgg_gc)
summary_vgg_gc

num_samples: 500


,infidelity,faithfulness_correlation,monotonicity_correlation,faithfulness_estimate,max_sensitivity,sparseness,random_logit
SoftPullback,54.714±79.396,0.056±0.106,0.463±0.463,0.553±0.212,0.258±0.072,0.568±0.04,-0.053±0.314
GuidedGradCam,1.99e+05±4.59e+05,0.029±0.107,0.516±0.444,0.1±0.107,0.356±0.098,0.827±0.063,0.428±0.283


In [36]:
# summary_resnet_gc.loc["GuidedGradCam"]

In [37]:
loaded_df_resnet = QuantusEvaluator.load_results(f"results/quantus_resnet_20_50")
print("num_samples:", len(loaded_df_resnet.iloc[0].iloc[0]))
summary_resnet = QuantusEvaluator.summarize_results(loaded_df_resnet)
# summary_resnet.loc[len(summary_resnet)] = summary_resnet_gc.loc["GuidedGradCam"]
summary_resnet.loc["GuidedGradCam"] = summary_resnet_gc.loc["GuidedGradCam"]
summary_resnet

num_samples: 1000


,faithfulness_correlation,monotonicity_correlation,faithfulness_estimate,pixel_flipping,infidelity,avg_sensitivity,max_sensitivity,sparseness,random_logit
SoftPullback,0.101±0.107,0.531±0.436,0.547±0.165,0.063±0.209,65.005±53.258,0.149±0.042,0.157±0.045,0.625±0.054,-0.075±0.386
DoublePullback,0.1±0.107,0.534±0.445,0.547±0.165,0.064±0.211,64.504±52.994,0.148±0.043,0.157±0.045,0.627±0.054,-0.072±0.386
DoublePullbackBis,0.105±0.11,0.527±0.435,0.546±0.164,0.065±0.213,64.143±52.833,0.148±0.042,0.157±0.046,0.629±0.055,-0.07±0.386
PullbackAscent,0.069±0.106,0.48±0.446,0.555±0.168,0.119±0.286,3.66e+05±4.42e+05,0.369±0.1,0.405±0.105,0.644±0.046,0.163±0.173
PullbackAscentBis,0.055±0.099,0.443±0.446,0.521±0.188,0.127±0.293,1.02e+06±2.58e+06,0.529±0.116,0.566±0.12,0.63±0.045,0.155±0.116
Gradient,0.015±0.105,0.26±0.46,0.189±0.171,0.021±0.108,597.745±896.168,1.075±0.149,1.12±0.16,0.548±0.037,-0.038±0.233
GradientShap,-0.015±0.103,0.403±0.473,-0.184±0.174,0.042±0.158,1725.292±3984.327,1.113±0.182,1.441±0.386,0.612±0.048,-0.031±0.243
IntegratedGradients,-0.016±0.102,0.431±0.484,-0.203±0.181,0.043±0.162,720.485±914,0.929±0.192,0.97±0.205,0.611±0.048,-0.034±0.236
Saliency,0.006±0.101,0.28±0.508,-0.145±0.435,0.03±0.131,5.89e+07±8.28e+07,0.739±0.103,0.774±0.113,0.479±0.044,0.412±0.096
DeepLift,-0.014±0.105,0.323±0.521,-0.185±0.173,0.043±0.163,429.665±675.871,1.091±0.155,1.144±0.168,0.62±0.049,-0.036±0.245


In [38]:
loaded_df_vgg = QuantusEvaluator.load_results(f"results/quantus_vgg_20_25")
print("num_samples:", len(loaded_df_vgg.iloc[0].iloc[0]))
summary_vgg = QuantusEvaluator.summarize_results(loaded_df_vgg)
summary_vgg.loc["GuidedGradCam"] = summary_vgg_gc.loc["GuidedGradCam"]
summary_vgg

num_samples: 500


,faithfulness_correlation,monotonicity_correlation,faithfulness_estimate,pixel_flipping,infidelity,avg_sensitivity,max_sensitivity,sparseness,random_logit
SoftPullback,0.064±0.109,0.449±0.461,0.553±0.212,0.005±0.031,54.158±76.998,0.25±0.069,0.257±0.071,0.568±0.04,-0.053±0.314
DoublePullback,0.06±0.105,0.45±0.457,0.562±0.206,0.007±0.041,42.929±60.507,0.241±0.06,0.25±0.063,0.573±0.04,-0.016±0.302
DoublePullbackBis,0.061±0.097,0.41±0.467,0.568±0.207,0.011±0.054,35.082±50.131,0.243±0.06,0.255±0.064,0.575±0.039,0.023±0.275
PullbackAscent,0.047±0.103,0.334±0.445,0.632±0.183,0.082±0.204,2.35e+05±6.66e+05,0.496±0.113,0.516±0.115,0.555±0.034,0.106±0.117
PullbackAscentBis,0.032±0.102,0.259±0.404,0.581±0.201,0.114±0.25,2.17e+06±5.08e+06,0.67±0.126,0.691±0.127,0.543±0.033,0.106±0.074
Gradient,0.017±0.099,0.264±0.473,0.158±0.161,0.005±0.039,378.156±472.738,0.987±0.124,1.008±0.127,0.521±0.037,-0.043±0.269
GradientShap,-0.018±0.102,0.285±0.499,-0.201±0.167,0.014±0.079,1251.2±2708.29,0.979±0.136,1.194±0.193,0.591±0.054,-0.034±0.253
IntegratedGradients,-0.014±0.095,0.343±0.491,-0.211±0.17,0.014±0.08,632.304±772.598,0.842±0.128,0.859±0.128,0.594±0.054,-0.039±0.259
Saliency,0.001±0.095,0.4±0.468,-0.007±0.401,0.008±0.058,5.40e+07±6.21e+07,0.645±0.072,0.66±0.075,0.442±0.045,0.411±0.106
DeepLift,-0.023±0.102,0.243±0.531,-0.199±0.16,0.014±0.083,317.814±375.703,1.004±0.129,1.03±0.133,0.597±0.053,-0.041±0.271


In [39]:
loaded_df_pvt = QuantusEvaluator.load_results(f"results/quantus_pvt_20_50")
print("num_samples:", len(loaded_df_pvt.iloc[0].iloc[0]))
summary_pvt = QuantusEvaluator.summarize_results(loaded_df_pvt)
summary_pvt

num_samples: 1000


,faithfulness_correlation,monotonicity_correlation,faithfulness_estimate,pixel_flipping,infidelity,avg_sensitivity,max_sensitivity,sparseness,random_logit
SoftPullback,0.028±0.099,-0.048±0.518,0.382±0.255,0.057±0.177,8.605±10.836,1.171±0.162,1.224±0.181,0.6±0.043,0.011±0.391
DoublePullback,0.031±0.104,0.132±0.499,0.505±0.292,0.047±0.152,4.311±4.19,1.12±0.09,1.163±0.103,0.545±0.029,0.081±0.088
DoublePullbackBis,0.034±0.102,0.188±0.489,0.498±0.299,0.046±0.148,3.913±3.992,1.086±0.077,1.123±0.086,0.547±0.03,0.078±0.074
PullbackAscent,0.044±0.107,0.111±0.423,0.502±0.343,0.122±0.216,6.04e+05±1.95e+06,0.953±0.094,0.973±0.094,0.483±0.026,0.135±0.117
PullbackAscentBis,0.035±0.103,0.106±0.418,0.387±0.352,0.203±0.273,4.96e+06±1.11e+07,0.923±0.092,0.942±0.093,0.473±0.029,0.144±0.081
Gradient,0.053±0.104,0.047±0.511,0.549±0.209,0.058±0.157,48.598±359.265,1.05±0.386,1.252±0.698,0.557±0.044,-0.002±0.202
GradientShap,-0.03±0.104,0.073±0.508,-0.354±0.267,0.134±0.261,856.61±5539.673,1.609±0.439,2.607±1.351,0.602±0.049,-0.003±0.207
IntegratedGradients,-0.032±0.107,0.083±0.508,-0.369±0.262,0.133±0.26,121.604±126.797,1.436±0.359,1.592±0.471,0.604±0.046,-0.005±0.206
Saliency,-0.009±0.104,0.15±0.507,0.059±0.371,0.136±0.244,2.07e+07±4.08e+07,0.771±0.115,0.837±0.272,0.545±0.06,0.56±0.113
DeepLift,-0.027±0.102,0.004±0.517,-0.392±0.252,0.126±0.253,69.056±247.563,1.095±0.387,1.285±0.684,0.624±0.051,0.002±0.221


In [40]:
short_explainers_map = {
    "Image": "Image",
    "SoftPullback": "SP (Ours)",
    "DoublePullback": "DP (Ours)",
    "DoublePullbackBis": "DPb (Ours)",
    "PullbackAscent": "PA (Ours)",
    "PullbackAscentBis": "PAb (Ours)",
    "Gradient": "Grad",
    "GuidedGradCam": "GG-CAM",
    "GradientShap": "GradShap",
    "DeepLift": "DeepLift",
    "Deconvolution": "Deconv",
    "IntegratedGradients": "IG",
    "Saliency": "Saliency",
    "InputXGradient": "IxG",
    # "LayerGradCam": "LGC",
    # "Occlusion": "Occ",
    # "KernelShap": "KShap",
    # "DeepLiftShap": "DeepLiftShap",
    # "FeatureAblation": "FeatAbla",
    # "FeaturePermutation": "FeatPerm",
    # "InternalInfluence": "IntInf",
    # "LRP": "LRP",
}
short_explainers_map = {
    "infidelity": "Infidelity",
    "faithfulness_correlation": "Faith.Corr",
    "monotonicity_correlation": "Mono.Corr",
    "faithfulness_estimate": "Faith.Est",
    "max_sensitivity": "Max.Sens",
    "random_logit": "Rand.Logit",
}

In [41]:

selected_columns = list(short_explainers_map.keys())
# filtered_df = summary_resnet[selected_columns].rename(columns=short_titles_map)
filtered_df = summary_resnet[selected_columns].rename(index=short_explainers_map, columns=short_explainers_map)
filtered_df.to_csv('results/summary_resnet_20_50.csv', index=True)
filtered_df

,Infidelity,Faith.Corr,Mono.Corr,Faith.Est,Max.Sens,Rand.Logit
SoftPullback,65.005±53.258,0.101±0.107,0.531±0.436,0.547±0.165,0.157±0.045,-0.075±0.386
DoublePullback,64.504±52.994,0.1±0.107,0.534±0.445,0.547±0.165,0.157±0.045,-0.072±0.386
DoublePullbackBis,64.143±52.833,0.105±0.11,0.527±0.435,0.546±0.164,0.157±0.046,-0.07±0.386
PullbackAscent,3.66e+05±4.42e+05,0.069±0.106,0.48±0.446,0.555±0.168,0.405±0.105,0.163±0.173
PullbackAscentBis,1.02e+06±2.58e+06,0.055±0.099,0.443±0.446,0.521±0.188,0.566±0.12,0.155±0.116
Gradient,597.745±896.168,0.015±0.105,0.26±0.46,0.189±0.171,1.12±0.16,-0.038±0.233
GradientShap,1725.292±3984.327,-0.015±0.103,0.403±0.473,-0.184±0.174,1.441±0.386,-0.031±0.243
IntegratedGradients,720.485±914,-0.016±0.102,0.431±0.484,-0.203±0.181,0.97±0.205,-0.034±0.236
Saliency,5.89e+07±8.28e+07,0.006±0.101,0.28±0.508,-0.145±0.435,0.774±0.113,0.412±0.096
DeepLift,429.665±675.871,-0.014±0.105,0.323±0.521,-0.185±0.173,1.144±0.168,-0.036±0.245


In [42]:

selected_columns = list(short_explainers_map.keys())
# filtered_df = summary_resnet[selected_columns].rename(columns=short_titles_map)
filtered_df = summary_vgg[selected_columns].rename(index=short_explainers_map, columns=short_explainers_map)
filtered_df.to_csv('results/summary_vgg_20_25.csv', index=True)
filtered_df

,Infidelity,Faith.Corr,Mono.Corr,Faith.Est,Max.Sens,Rand.Logit
SoftPullback,54.158±76.998,0.064±0.109,0.449±0.461,0.553±0.212,0.257±0.071,-0.053±0.314
DoublePullback,42.929±60.507,0.06±0.105,0.45±0.457,0.562±0.206,0.25±0.063,-0.016±0.302
DoublePullbackBis,35.082±50.131,0.061±0.097,0.41±0.467,0.568±0.207,0.255±0.064,0.023±0.275
PullbackAscent,2.35e+05±6.66e+05,0.047±0.103,0.334±0.445,0.632±0.183,0.516±0.115,0.106±0.117
PullbackAscentBis,2.17e+06±5.08e+06,0.032±0.102,0.259±0.404,0.581±0.201,0.691±0.127,0.106±0.074
Gradient,378.156±472.738,0.017±0.099,0.264±0.473,0.158±0.161,1.008±0.127,-0.043±0.269
GradientShap,1251.2±2708.29,-0.018±0.102,0.285±0.499,-0.201±0.167,1.194±0.193,-0.034±0.253
IntegratedGradients,632.304±772.598,-0.014±0.095,0.343±0.491,-0.211±0.17,0.859±0.128,-0.039±0.259
Saliency,5.40e+07±6.21e+07,0.001±0.095,0.4±0.468,-0.007±0.401,0.66±0.075,0.411±0.106
DeepLift,317.814±375.703,-0.023±0.102,0.243±0.531,-0.199±0.16,1.03±0.133,-0.041±0.271


In [43]:

selected_columns = list(short_explainers_map.keys())
# filtered_df = summary_resnet[selected_columns].rename(columns=short_titles_map)
filtered_df = summary_pvt[selected_columns].rename(index=short_explainers_map, columns=short_explainers_map)
filtered_df.to_csv('results/summary_pvt_20_50.csv', index=True)
filtered_df

,Infidelity,Faith.Corr,Mono.Corr,Faith.Est,Max.Sens,Rand.Logit
SoftPullback,8.605±10.836,0.028±0.099,-0.048±0.518,0.382±0.255,1.224±0.181,0.011±0.391
DoublePullback,4.311±4.19,0.031±0.104,0.132±0.499,0.505±0.292,1.163±0.103,0.081±0.088
DoublePullbackBis,3.913±3.992,0.034±0.102,0.188±0.489,0.498±0.299,1.123±0.086,0.078±0.074
PullbackAscent,6.04e+05±1.95e+06,0.044±0.107,0.111±0.423,0.502±0.343,0.973±0.094,0.135±0.117
PullbackAscentBis,4.96e+06±1.11e+07,0.035±0.103,0.106±0.418,0.387±0.352,0.942±0.093,0.144±0.081
Gradient,48.598±359.265,0.053±0.104,0.047±0.511,0.549±0.209,1.252±0.698,-0.002±0.202
GradientShap,856.61±5539.673,-0.03±0.104,0.073±0.508,-0.354±0.267,2.607±1.351,-0.003±0.207
IntegratedGradients,121.604±126.797,-0.032±0.107,0.083±0.508,-0.369±0.262,1.592±0.471,-0.005±0.206
Saliency,2.07e+07±4.08e+07,-0.009±0.104,0.15±0.507,0.059±0.371,0.837±0.272,0.56±0.113
DeepLift,69.056±247.563,-0.027±0.102,0.004±0.517,-0.392±0.252,1.285±0.684,0.002±0.221


# Older runs for 500 examples and less explainers:

In [45]:
loaded_df_resnet = QuantusEvaluator.load_results(f"results/quantus_resnet_20_25_model_name_resnet50")
print("num_samples:", len(loaded_df_resnet.iloc[0].iloc[0]))
summary_resnet = QuantusEvaluator.summarize_results(loaded_df_resnet)
summary_resnet

num_samples: 500


,faithfulness_correlation,monotonicity_correlation,faithfulness_estimate,pixel_flipping,infidelity,avg_sensitivity,max_sensitivity,sparseness,random_logit
SoftPullback,0.11±0.105,0.546±0.416,0.568±0.171,0.057±0.2,60.827±50.344,0.171±0.052,0.18±0.055,0.616±0.052,-0.08±0.355
DoublePullback,0.107±0.104,0.536±0.423,0.569±0.17,0.059±0.203,59.977±49.877,0.17±0.052,0.18±0.056,0.618±0.052,-0.077±0.355
Gradient,0.011±0.103,0.277±0.463,0.187±0.175,0.02±0.105,589.494±777.132,1.084±0.155,1.129±0.165,0.548±0.036,-0.042±0.236
GradientShap,-0.016±0.102,0.427±0.467,-0.191±0.173,0.041±0.154,1535.851±2748.603,1.136±0.19,1.374±0.293,0.613±0.049,-0.031±0.253
IntegratedGradients,-0.019±0.102,0.424±0.486,-0.208±0.183,0.043±0.161,735.057±978.154,0.942±0.196,0.983±0.209,0.612±0.049,-0.034±0.245
Saliency,0.005±0.102,0.318±0.505,-0.126±0.435,0.03±0.129,5.64e+07±8.77e+07,0.745±0.109,0.78±0.12,0.48±0.042,0.41±0.095
DeepLift,-0.012±0.099,0.292±0.519,-0.186±0.171,0.041±0.158,414.154±605.901,1.102±0.166,1.158±0.182,0.62±0.049,-0.036±0.251
InputXGradient,-0.016±0.099,0.312±0.513,-0.183±0.17,0.041±0.158,416.579±591.101,1.101±0.165,1.156±0.177,0.62±0.049,-0.036±0.25
Deconvolution,0.005±0.099,0.421±0.413,0.222±0.247,0.011±0.075,6.97e+09±5.07e+09,0.707±0.119,0.726±0.118,0.542±0.008,1±4.33e-05


In [46]:
loaded_df_vgg = QuantusEvaluator.load_results(f"results/quantus_vgg_20_25_vgg11_bn")
print("num_samples:", len(loaded_df_vgg.iloc[0].iloc[0]))
summary_vgg = QuantusEvaluator.summarize_results(loaded_df_vgg)
summary_vgg

num_samples: 500


,faithfulness_correlation,monotonicity_correlation,faithfulness_estimate,pixel_flipping,infidelity,avg_sensitivity,max_sensitivity,sparseness,random_logit
SoftPullback,0.069±0.11,0.463±0.455,0.565±0.214,0.005±0.028,54.736±69.833,0.253±0.065,0.26±0.067,0.568±0.041,-0.05±0.305
DoublePullback,0.061±0.106,0.455±0.452,0.573±0.21,0.007±0.039,43.377±53.441,0.247±0.058,0.256±0.061,0.572±0.041,-0.016±0.29
Gradient,0.014±0.101,0.23±0.485,0.158±0.161,0.005±0.039,367.498±480.932,0.987±0.124,1.009±0.127,0.521±0.037,-0.043±0.269
GradientShap,-0.018±0.103,0.322±0.49,-0.201±0.167,0.014±0.078,1250.731±2906.688,0.986±0.133,1.141±0.179,0.591±0.054,-0.035±0.253
IntegratedGradients,-0.025±0.1,0.345±0.486,-0.211±0.17,0.014±0.08,639.19±800.556,0.842±0.128,0.858±0.128,0.594±0.054,-0.039±0.259
Saliency,0.009±0.104,0.418±0.457,-0.007±0.401,0.008±0.058,5.04e+07±5.84e+07,0.645±0.072,0.661±0.076,0.442±0.045,0.411±0.106
DeepLift,-0.02±0.103,0.214±0.534,-0.199±0.16,0.014±0.083,317.133±381.703,1.004±0.129,1.031±0.132,0.597±0.053,-0.041±0.271
InputXGradient,-0.008±0.094,0.234±0.535,-0.199±0.16,0.014±0.083,316.903±370.885,1.004±0.129,1.031±0.132,0.597±0.053,-0.041±0.271
Deconvolution,0.002±0.097,0.005±0.481,-0.051±0.14,0.005±0.03,6.33e+07±4.87e+07,0.784±0.163,0.793±0.164,0.493±0.005,0.998±9.85e-04


In [47]:
loaded_df_pvt = QuantusEvaluator.load_results(f"results/quantus_pvt_20_25_pvt_v2_b1")
print("num_samples:", len(loaded_df_pvt.iloc[0].iloc[0]))
summary_pvt = QuantusEvaluator.summarize_results(loaded_df_pvt)
summary_pvt

num_samples: 500


,faithfulness_correlation,monotonicity_correlation,faithfulness_estimate,pixel_flipping,infidelity,avg_sensitivity,max_sensitivity,sparseness,random_logit
SoftPullback,0.03±0.099,-0.056±0.529,0.379±0.266,0.055±0.174,8.95±11.975,1.175±0.164,1.227±0.185,0.599±0.04,0.004±0.386
DoublePullback,0.029±0.106,0.139±0.499,0.5±0.3,0.044±0.146,4.283±4.072,1.116±0.09,1.159±0.101,0.546±0.027,0.08±0.087
Gradient,0.055±0.1,0.054±0.514,0.544±0.214,0.055±0.153,55.015±384.654,1.063±0.379,1.269±0.692,0.558±0.043,-0.002±0.203
GradientShap,-0.03±0.103,0.083±0.51,-0.353±0.269,0.131±0.257,769.192±3392.535,1.648±0.473,2.555±1.107,0.601±0.048,5.01e-04±0.205
IntegratedGradients,-0.039±0.105,0.086±0.514,-0.366±0.265,0.129±0.255,122.147±123.952,1.461±0.372,1.641±0.519,0.604±0.046,-0.006±0.21
Saliency,-0.01±0.099,0.151±0.505,0.065±0.369,0.133±0.241,2.16e+07±4.48e+07,0.771±0.109,0.838±0.266,0.545±0.058,0.559±0.112
DeepLift,-0.024±0.103,0.014±0.521,-0.392±0.255,0.125±0.251,72.227±290.518,1.104±0.376,1.305±0.684,0.625±0.05,0.002±0.226
InputXGradient,-0.024±0.109,0.012±0.517,-0.392±0.254,0.124±0.251,73.768±291.517,1.097±0.36,1.302±0.649,0.625±0.05,0.002±0.226
Deconvolution,0.06±0.105,0.057±0.511,0.544±0.214,0.055±0.153,57.408±426.218,1.06±0.371,1.242±0.604,0.558±0.043,-0.002±0.203
